In [1]:
import pandas as pd
import numpy as np
import webbrowser
import os
from geopy.geocoders import ArcGIS
import time
import folium
import osmnx as ox
import networkx as nx
import pyproj
from shapely.geometry import Point

In [3]:
patient=pd.read_excel("patient_list.xlsx")
patient

,Patient Name,Age,Disease
0,Eleanor Vance,34,Migraine with Aura
1,Marcus Chen,52,Type 2 Diabetes
2,Aaliyah Jackson,28,Asthma
3,Liam O'Connor,67,Hypertension
4,Sophia Martinez,11,Acute Otitis Media
5,David Kim,45,GERD
6,Emma Watson,73,Osteoarthritis
7,Julius Novak,23,Infectious Mononucleosis
8,Chloe Dubois,29,Generalized Anxiety Disorder
9,Arjun Mehta,61,Chronic Kidney Disease (Stage 2)


In [18]:
G = ox.graph_from_place(
    "Greater Mumbai, Maharashtra, India",
    network_type="drive"
)

nodes, _ = ox.graph_to_gdfs(G)

sample = nodes.sample(n=len(patient), random_state=42)

addresses = sample[['y','x']].rename(
    columns={'y':'Latitude','x':'Longitude'}
)

In [19]:
idx = np.random.randint(0, len(addresses), size=len(patient))

patient["Latitude"] = addresses.iloc[idx]["Latitude"].values
patient["Longitude"] = addresses.iloc[idx]["Longitude"].values

In [20]:
patient

,Patient Name,Age,Disease,Latitude,Longitude
0,Eleanor Vance,34,Migraine with Aura,19.143896,72.995997
1,Marcus Chen,52,Type 2 Diabetes,19.458188,72.819044
2,Aaliyah Jackson,28,Asthma,19.251862,72.870831
3,Liam O'Connor,67,Hypertension,19.181006,73.051889
4,Sophia Martinez,11,Acute Otitis Media,19.024750,73.018203
5,David Kim,45,GERD,19.254993,73.032868
6,Emma Watson,73,Osteoarthritis,18.884484,72.914845
7,Julius Novak,23,Infectious Mononucleosis,19.254993,73.032868
8,Chloe Dubois,29,Generalized Anxiety Disorder,19.251862,72.870831
9,Arjun Mehta,61,Chronic Kidney Disease (Stage 2),19.161230,72.786317


In [65]:
def create_map(df,graph,graph_proj):
    tiles ="https://{s}.basemaps.cartocdn.com/rastertiles/voyager_nolabels/{z}/{x}/{y}{r}.png"
    latitude=(df['Latitude'].max()+df['Latitude'].min())/2
    longitude=(df['Longitude'].max()+df['Longitude'].min())/2

    m = folium.Map(
         location=[latitude,longitude],
         zoom_start=10,
         tiles=tiles,
         attr='© OpenStreetMap © CARTO'
    )
    grouped = df.groupby(["Latitude", "Longitude"])
    count_df=grouped.size().reset_index(name="Count")
    count_lookup = count_df.set_index(["Latitude", "Longitude"])["Count"]

    min_value = count_df["Count"].min()
    max_value = count_df["Count"].max()
    for (lat, lon), group in grouped:
       count = count_lookup.loc[(lat, lon)]

       if max_value == min_value:
          opacity = 0.8
       else:
        opacity = (count - min_value) / (max_value - min_value)
       popup = "<br>".join(
        f"{row['Patient Name']} ({row['Age']} yrs, {row['Disease']})"
        for _, row in group.iterrows()
    )
       folium.CircleMarker(
        location=[lat, lon],
        radius=8+len(group),
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity= 1-opacity * 0.7,
        popup=folium.Popup(popup, max_width=350)
    ).add_to(m)
       folium.Marker(
    location=[lat,lon],
    icon=folium.DivIcon(
        icon_size=(60, 30),
        icon_anchor=(-10, 5),
        html=f"""
        <div style="
            font-size:10px;
            font-weight:bold;
            color:black;
            background-color:white;
            border:1px solid black;
            border-radius:4px;
            padding:1px 4px;
            white-space: nowrap;
        ">
        {len(group)} patient(s)
        </div>
        """
    )
).add_to(m)

    file_name = "patient_mapping.html"
    m.save(file_name)  

    # Open the generated map in the default web browser             
    webbrowser.open('file://' + os.path.realpath(file_name)) 


In [66]:
padding = 0.03                                                 

    # Determine geographic bounding box limits
north =patient["Latitude"].max() + padding
south =patient["Latitude"].min() - padding
east =patient["Longitude"].max() + padding
west =patient["Longitude"].min() - padding 
    
    # Download drivable road network within the bounding box
graph = ox.graph_from_bbox(bbox=(west, south, east, north), network_type="drive")

    # Verify that a valid road network was retrieved
if len(graph) == 0:
        raise ValueError("No drivable roads found in this specific coordinate region.")
    
    # Project graph to a suitable coordinate reference system for accurate distance calculations
graph_proj = ox.project_graph(graph)
create_map(patient,graph,graph_proj)